# Analysis 1 · Subdomains over time

Subdomain structure across the 15 federal domains, from `surthost_seg_2`
(subdomain-1, e.g. `data` in `data.doi.gov`) and `surthost_seg_3` (subdomain-2).
Deduplicated to one unique URL per `(crawl_year, surtkey)`, `dns:` removed; SURT
strips `www`, so the apex is labeled `(bare/www)`.

Companion: `a2_paths.ipynb`. Outputs → `analysis/deliverables/`.

In [ ]:
import sys, os
os.umask(0o000)   # everything this notebook writes is world read/write (shared server, open to all)
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import config
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 90)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']

def _writable_dir(preferred='deliverables'):
    for cand in (preferred, os.path.expanduser('~/eot_deliverables')):
        try:
            os.makedirs(cand, exist_ok=True)
            try: os.chmod(cand, 0o777)
            except OSError: pass
            t = os.path.join(cand, '.wtest'); open(t,'w').close(); os.remove(t)
            return cand
        except OSError:
            continue
    raise RuntimeError('no writable output dir found')
OUTDIR = _writable_dir()
print('OUTDIR =', os.path.abspath(OUTDIR))

DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

def per_domain(sql):
    frames = []
    for dom, path in DBS.items():
        con = duckdb.connect(str(path), read_only=True)
        df = con.sql(sql).df(); con.close()
        df.insert(0, 'domain', dom); frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    if 'crawl_year' in out.columns:
        out['crawl_year'] = out['crawl_year'].astype(str)
    return out

def year_cols(df):
    return [y for y in YEAR_ORDER if y in df.columns]

def save(name):
    '''Output path; remove any stale file first so writes never hit the overwrite perm issue.'''
    p = os.path.join(OUTDIR, name)
    try:
        if os.path.exists(p): os.remove(p)
    except OSError: pass
    return p

## Load subdomain counts (all domains)

In [ ]:
SUBDOMAIN_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey,
         min(surthost_seg_2) AS sub1,
         min(surthost_seg_3) AS sub2
  FROM eot_captures
  WHERE url NOT LIKE 'dns:%'
  GROUP BY 1, 2
)
SELECT COALESCE(sub1, '(bare/www)') AS subdomain_1,
       sub2                         AS subdomain_2,
       crawl_year, count(*)         AS n
FROM dedup GROUP BY 1, 2, 3
'''
subs = per_domain(SUBDOMAIN_SQL)
subs.to_csv(save('a1_subdomains_long.csv'), index=False)
print(f"{len(subs):,} (domain, subdomain_1, subdomain_2, year) rows")
subs.head()

## 1a. Headline — distinct subdomain-1 count per domain per year

In [ ]:
distinct1 = (subs.groupby(['domain','crawl_year'])['subdomain_1'].nunique()
                 .reset_index(name='distinct_subdomains'))
d1_piv = distinct1.pivot(index='domain', columns='crawl_year', values='distinct_subdomains')
d1_piv = d1_piv[year_cols(d1_piv)].reindex([d for d in config.TARGET_DOMAINS if d in d1_piv.index])
d1_piv.to_csv(save('a1_distinct_subdomain_counts.csv'))
display(d1_piv.fillna(0).astype(int))

fig, ax = plt.subplots(figsize=(11,6))
for dom in d1_piv.index:
    ax.plot(d1_piv.columns, d1_piv.loc[dom].values, 'o-', alpha=.6, label=dom)
ax.set_xlabel('crawl year'); ax.set_ylabel('distinct subdomain-1 labels')
ax.set_title('Number of distinct subdomains over time, by domain'); ax.legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.savefig(save('a1_distinct_subdomains.png'), dpi=150, bbox_inches='tight'); plt.show()

## 1b. Subdomain-1 URL counts per year, per domain (pivot tables)

One CSV per domain; doi shown inline.

In [ ]:
sub1_counts = subs.groupby(['domain','subdomain_1','crawl_year'])['n'].sum().reset_index()
for dom, g in sub1_counts.groupby('domain'):
    piv = g.pivot_table(index='subdomain_1', columns='crawl_year', values='n', aggfunc='sum').fillna(0).astype(int)
    piv = piv[year_cols(piv)]; piv['total'] = piv.sum(axis=1)
    piv.sort_values('total', ascending=False).to_csv(save(f"a1_subdomain1_{dom.replace('.gov','')}.csv"))
print('wrote per-domain subdomain-1 pivots to', OUTDIR)

example = 'doi.gov' if 'doi.gov' in DBS else list(DBS)[0]
g = sub1_counts[sub1_counts.domain==example]
ex = g.pivot_table(index='subdomain_1', columns='crawl_year', values='n', aggfunc='sum').fillna(0).astype(int)
ex = ex[year_cols(ex)]; ex['total'] = ex.sum(axis=1)
print(f"Example — {example}"); ex.sort_values('total', ascending=False).head(25)

## 1c. Subdomain-2 within subdomain-1 (`surthost_seg_3`)

In [ ]:
sub2 = subs[subs.subdomain_2.notna()].copy()
sub2_counts = sub2.groupby(['domain','subdomain_1','subdomain_2','crawl_year'])['n'].sum().reset_index()
sub2_counts.to_csv(save('a1_subdomain2_long.csv'), index=False)
if len(sub2_counts):
    piv2 = sub2_counts.pivot_table(index=['domain','subdomain_1','subdomain_2'],
                                   columns='crawl_year', values='n', aggfunc='sum').fillna(0).astype(int)
    piv2 = piv2[year_cols(piv2)]; piv2['total'] = piv2.sum(axis=1)
    display(piv2.sort_values('total', ascending=False).head(30))
else:
    print('No second-level subdomains found.')

## 1d. Small multiples — top subdomains per domain over time

One panel per domain; each line is a subdomain's URL count across crawl years
(log scale). Lets you compare *shapes* of subdomain growth across agencies.

In [ ]:
import math
TOP_PER = 6
doms = [d for d in config.TARGET_DOMAINS if d in subs.domain.unique()]
ncol = 3; nrow = math.ceil(len(doms)/ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 2.8*nrow)); axes = np.array(axes).flatten()
for ax, dom in zip(axes, doms):
    d = subs[subs.domain==dom]
    top = d.groupby('subdomain_1')['n'].sum().nlargest(TOP_PER).index
    for sd in top:
        s = d[d.subdomain_1==sd].groupby('crawl_year')['n'].sum().reindex(YEAR_ORDER).fillna(0)
        ax.plot(s.index, s.values.clip(min=0.5), 'o-', ms=3, label=str(sd)[:14])
    ax.set_title(dom, fontsize=9); ax.set_yscale('log'); ax.tick_params(labelsize=6)
    ax.legend(fontsize=5, ncol=2, loc='upper left')
for ax in axes[len(doms):]: ax.axis('off')
plt.suptitle(f'Top {TOP_PER} subdomains per domain over time (URL counts, log)', y=1.005)
plt.tight_layout(); plt.savefig(save('a1_small_multiples.png'), dpi=150, bbox_inches='tight'); plt.show()

## 1e. Cross-domain — shared subdomain naming conventions

Which subdomain-1 labels recur across *multiple* agencies? Shared names (e.g.
`search`, `apps`, `data`, `blog`) are infrastructure conventions; domain-unique
labels are agency-specific. A pattern both within and across .gov.

In [ ]:
shared = (subs[subs.subdomain_1 != '(bare/www)']
          .groupby('subdomain_1')['domain'].nunique().sort_values(ascending=False))
shared = shared[shared >= 2]
shared.to_frame('n_domains').to_csv(save('a1_shared_subdomain_names.csv'))
print(f"{len(shared)} subdomain labels appear in 2+ domains. Most widely shared:")
top = shared.head(20)
for label, nd in top.items():
    doms_with = sorted(subs[subs.subdomain_1==label]['domain'].unique())
    print(f"  {label:18s} {nd:2d} domains  ({', '.join(d.replace('.gov','') for d in doms_with)})")

## 1f. Track one subdomain over time (e.g. `fns`)

Has a specific subdomain changed over time, and does it appear in more than one
agency? Change the label to explore (e.g. `fns`, `data`, `search`, `apps`).

In [ ]:
def subdomain_trend(label):
    d = subs[subs.subdomain_1 == label]
    if d.empty:
        print(f"'{label}' not present in the loaded domains."); return
    piv = d.pivot_table(index='domain', columns='crawl_year', values='n', aggfunc='sum').fillna(0).astype(int)
    piv = piv[year_cols(piv)]
    display(piv)
    piv.T.plot(marker='o', figsize=(9,5))
    plt.title(f"'{label}.*' subdomain — unique-URL counts over time")
    plt.ylabel('unique URLs'); plt.xlabel('crawl year'); plt.tight_layout(); plt.show()

subdomain_trend('fns')

## 1g. QA — sample URLs per subdomain (`url` + `surtkey`)

First 5 URLs for every subdomain per year, for manual review. Writes
`a1_subdomain_url_samples.csv` — the file for the co-PI to eyeball.

In [ ]:
SAMPLE_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey,
         min(url)                                    AS url,
         min(COALESCE(surthost_seg_2, '(bare/www)')) AS subdomain_1,
         min(surthost_seg_3)                         AS subdomain_2
  FROM eot_captures
  WHERE url NOT LIKE 'dns:%'
  GROUP BY 1, 2
),
ranked AS (
  SELECT crawl_year, subdomain_1, subdomain_2, url, surtkey,
         row_number() OVER (PARTITION BY subdomain_1, crawl_year ORDER BY url) AS rn
  FROM dedup
)
SELECT subdomain_1, subdomain_2, crawl_year, rn, url, surtkey
FROM ranked WHERE rn <= 5
ORDER BY subdomain_1, crawl_year, rn
'''
samples = per_domain(SAMPLE_SQL)[['domain','subdomain_1','subdomain_2','crawl_year','rn','url','surtkey']]
samples.to_csv(save('a1_subdomain_url_samples.csv'), index=False)
print(f"{len(samples):,} sample rows across {samples.domain.nunique()} domains -> a1_subdomain_url_samples.csv")
samples.head(20)